In [1]:
import librosa
import numpy as np
from spafe.features.lfcc import lfcc
from spafe.utils.preprocessing import SlidingWindow

In [2]:
def extract_lfcc(
    wav_path,
    sr=16000,
    num_ceps=20,
    nfilts=46,
    nfft=400,        # 25 ms @16k
    win_len=0.025,
    win_hop=0.01
):
    signal, sr = librosa.load(wav_path, sr=sr)
    window = SlidingWindow(
        win_len=win_len,
        win_hop=win_hop,
        win_type="hamming"
    )
    lfcc_feat = lfcc(
        sig=signal,
        fs=sr,
        num_ceps=num_ceps,
        nfilts=nfilts,
        nfft=nfft,
        window=window
    )
    lfcc_delta = librosa.feature.delta(lfcc_feat)

    # delta-delta
    lfcc_delta2 = librosa.feature.delta(lfcc_feat, order=2)

    # ghép → LFCC-60
    lfcc_60 = np.hstack([lfcc_feat, lfcc_delta, lfcc_delta2])
    
    return lfcc_60   # shape: (num_frames, num_ceps)


In [5]:
def build_lfcc_dataset(
    wav_list,
    sr=16000,
    out_path="X_mfcc.npy"
):
    all_feats = []

    for wav_path in wav_list:
        print(wav_path)
        lfcc = extract_lfcc(wav_path, sr)
        all_feats.append(lfcc)

        print(f"{wav_path} -> {lfcc.shape}")

    # Gộp theo frame
    X = np.vstack(all_feats)
    #X = X[:int(X.shape[0]//10), :]
    np.save(out_path, X)
    print("Saved:", out_path, X.shape)

    return X


In [7]:
DIR = "E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed"

wav_list = [f"{DIR}/{i + 1}.wav" for i in range(180)]
# wav_list.pop(766)

X_mfcc = build_lfcc_dataset(wav_list, out_path="E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train_mini/aurora_lfcc_speech_mixed.npy")


E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/1.wav
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/1.wav -> (268, 60)
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/2.wav
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/2.wav -> (259, 60)
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/3.wav
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/3.wav -> (218, 60)
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/4.wav
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/4.wav -> (206, 60)
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/5.wav
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/5.wav -> (214, 60)
E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed/6.wav
